# PathSnap — Initial XGBoost experiment

**Status: blocked on the team's shared candidate-feature dataset and preprocessing/evaluation utilities.** This notebook intentionally does not generate candidates, assign labels, engineer an independent feature set, or train on raw trip files. Raw-data inspection below is limited to format, inventory, timestamps, and integrity checks. No model results are reported until the approved shared inputs arrive.


## 1. Imports and configuration

Paths are relative to the workspace root (the directory containing this notebook). Set the prepared-feature path and explicitly approved feature-column list after receiving them from the team.

In [3]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, log_loss,
)

WORKSPACE_ROOT = Path.cwd()
RAW_DATA_ROOT = WORKSPACE_ROOT / "Student folder" / "Student folder"
TRAIN_DIR = RAW_DATA_ROOT / "train"
STUDENT_TEST_DIR = RAW_DATA_ROOT / "student_test"
# Fill in these separately to preserve the train/student-test boundary.
SHARED_TRAIN_FEATURES_PATH = WORKSPACE_ROOT / "preprocessed_data" / "train_candidates.csv"
SHARED_STUDENT_TEST_FEATURES_PATH = WORKSPACE_ROOT / "preprocessed_data" / "student_test_candidates.csv"
# These must come from the team's agreed feature definitions; never infer them from all columns.
TEAM_FEATURE_COLUMNS = [
    "distance_to_segment_m",
    "projection_fraction",
    "observation_speed_mps",
    "heading_difference_rad",
]
TRIP_COLUMN = "trip_id"
OBSERVATION_COLUMN = "observation_id"
CANDIDATE_COLUMN = "candidate_id"
LABEL_COLUMN = "label"
N_SPLITS = 5
RANDOM_STATE = 42
OUTPUT_DIR = WORKSPACE_ROOT / "outputs" / "xgboost"

XGB_CONFIG = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

print(f"Workspace: {WORKSPACE_ROOT}")
print(f"XGBoost version: {xgboost.__version__}")
print("Shared training features configured:", SHARED_TRAIN_FEATURES_PATH is not None)
print("Shared student-test features configured:", SHARED_STUDENT_TEST_FEATURES_PATH is not None)


Workspace: C:\Users\Shibi R Shetty\ML HACKTHON
XGBoost version: 3.4.1
Shared training features configured: True
Shared student-test features configured: True


## 2. Dataset inspection

The source files are headerless, whitespace-delimited records: `.track` has longitude, latitude, timestamp (seconds); `.nodes` has longitude and latitude; `.arcs` has from-node and to-node IDs; `.route` contains reference arc IDs. This cell inventories and checks raw files only; it does not associate observations with candidates or derive model labels.

In [5]:
def read_raw_table(path, expected_columns):
    frame = pd.read_csv(path, sep=r"\s+", header=None, dtype="float64")
    if frame.shape[1] != expected_columns:
        raise ValueError(f"{path} has {frame.shape[1]} columns; expected {expected_columns}")
    return frame

def inspect_partition(partition_dir, partition_name):
    if not partition_dir.is_dir():
        raise FileNotFoundError(f"Missing raw partition: {partition_dir}")
    trip_dirs = sorted(path for path in partition_dir.iterdir() if path.is_dir())
    records = []
    for trip_dir in trip_dirs:
        trip_id = trip_dir.name
        paths = {suffix: trip_dir / f"{trip_id}{suffix}" for suffix in (".track", ".nodes", ".arcs", ".route")}
        missing = [str(path) for path in paths.values() if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"Trip {trip_id} is missing files: {missing}")

        track = read_raw_table(paths[".track"], 3)
        nodes = read_raw_table(paths[".nodes"], 2)
        arcs = read_raw_table(paths[".arcs"], 2)
        route = read_raw_table(paths[".route"], 1).iloc[:, 0]
        timestamps = track.iloc[:, 2].to_numpy()
        gaps = np.diff(timestamps)
        finite_track = np.isfinite(track.to_numpy()).all(axis=1)
        finite_nodes = np.isfinite(nodes.to_numpy()).all(axis=1)
        valid_track_coordinates = (
            track.iloc[:, 0].between(-180, 180).to_numpy()
            & track.iloc[:, 1].between(-90, 90).to_numpy()
        )
        valid_node_coordinates = (
            nodes.iloc[:, 0].between(-180, 180).to_numpy()
            & nodes.iloc[:, 1].between(-90, 90).to_numpy()
        )
        valid_arc_endpoints = (
            np.isfinite(arcs.to_numpy()).all()
            and np.equal(arcs.to_numpy(), np.floor(arcs.to_numpy())).all()
            and (arcs.to_numpy() >= 0).all()
            and (arcs.to_numpy() < len(nodes)).all()
        )
        route_ids_valid = (
            np.isfinite(route.to_numpy()).all()
            and np.equal(route.to_numpy(), np.floor(route.to_numpy())).all()
            and (route.to_numpy() >= 0).all()
            and (route.to_numpy() < len(arcs)).all()
        )
        records.append({
            "partition": partition_name,
            "trip_id": trip_id,
            "track_rows": len(track),
            "node_rows": len(nodes),
            "arc_rows": len(arcs),
            "route_rows": len(route),
            "missing_values_track": int(track.isna().sum().sum()),
            "nonfinite_track_rows": int((~finite_track).sum()),
            "invalid_gps_coordinate_rows": int((~valid_track_coordinates).sum()),
            "nonfinite_node_rows": int((~finite_nodes).sum()),
            "invalid_node_coordinate_rows": int((~valid_node_coordinates).sum()),
            "timestamps_strictly_increasing": bool((gaps > 0).all()),
            "nonpositive_time_gaps": int((gaps <= 0).sum()),
            "irregular_time_gaps": int((~np.isclose(gaps, 1.0)).sum()),
            "min_time_gap_seconds": float(gaps.min()) if len(gaps) else np.nan,
            "max_time_gap_seconds": float(gaps.max()) if len(gaps) else np.nan,
            "valid_arc_endpoints": bool(valid_arc_endpoints),
            "route_arc_ids_in_range": bool(route_ids_valid),
            "file_bytes": sum(path.stat().st_size for path in paths.values()),
        })
    return pd.DataFrame.from_records(records)

inventory = pd.concat([
    inspect_partition(TRAIN_DIR, "train"),
    inspect_partition(STUDENT_TEST_DIR, "student_test"),
], ignore_index=True)
print("Trip counts by partition:")
display(inventory.groupby("partition").agg(
    trips=("trip_id", "nunique"), observations=("track_rows", "sum"),
    road_nodes=("node_rows", "sum"), arcs=("arc_rows", "sum"),
    reference_route_rows=("route_rows", "sum"), file_bytes=("file_bytes", "sum"),
))
print("\nRaw schema (representative trip):")
sample_trip = next(path for path in sorted(TRAIN_DIR.iterdir()) if path.is_dir())
for suffix, names, ncols in [
    (".track", ["longitude", "latitude", "timestamp_seconds"], 3),
    (".nodes", ["longitude", "latitude"], 2),
    (".arcs", ["from_node_id", "to_node_id"], 2),
    (".route", ["reference_arc_id"], 1),
]:
    sample = read_raw_table(sample_trip / f"{sample_trip.name}{suffix}", ncols)
    sample.columns = names
    print(f"{suffix}: {names}; sample rows={len(sample)}")
    display(sample.head(3))
print("\nTrip-level data-quality summary:")
display(inventory.groupby("partition").agg(
    trips=("trip_id", "count"), min_observations=("track_rows", "min"),
    max_observations=("track_rows", "max"), trips_with_nonpositive_gaps=("nonpositive_time_gaps", lambda s: int((s > 0).sum())),
    trips_with_irregular_gaps=("irregular_time_gaps", lambda s: int((s > 0).sum())),
    trips_with_missing_track_values=("missing_values_track", lambda s: int((s > 0).sum())),
    trips_with_nonfinite_track_rows=("nonfinite_track_rows", lambda s: int((s > 0).sum())),
    trips_with_invalid_gps_coordinates=("invalid_gps_coordinate_rows", lambda s: int((s > 0).sum())),
    trips_with_nonfinite_node_rows=("nonfinite_node_rows", lambda s: int((s > 0).sum())),
    trips_with_invalid_node_coordinates=("invalid_node_coordinate_rows", lambda s: int((s > 0).sum())),
    trips_with_invalid_arcs=("valid_arc_endpoints", lambda s: int((~s).sum())),
    trips_with_invalid_reference_arc_ids=("route_arc_ids_in_range", lambda s: int((~s).sum())),
))
print("\nAll trip IDs:")
for partition, group in inventory.groupby("partition", sort=False):
    print(f"{partition} ({len(group)}): {', '.join(group['trip_id'])}")


Trip counts by partition:
              trips  observations  ...  reference_route_rows  file_bytes
partition                          ...                                  
student_test     10         19160  ...                  3477     4368591
train            80        208417  ...                 31336    89737452

[2 rows x 6 columns]

Raw schema (representative trip):
.track: ['longitude', 'latitude', 'timestamp_seconds']; sample rows=1315
   longitude   latitude  timestamp_seconds
0   8.595203  49.844053                1.0
1   8.595294  49.844441                2.0
2   8.595352  49.844883                3.0
.nodes: ['longitude', 'latitude']; sample rows=88809
   longitude   latitude
0   8.581679  49.857880
1   8.584637  49.858295
2   8.580173  49.858567
.arcs: ['from_node_id', 'to_node_id']; sample rows=165790
   from_node_id  to_node_id
0           0.0       281.0
1           0.0         1.0
2           0.0         5.0
.route: ['reference_arc_id']; sample rows=525
   reference_ar

## 3. Loading prepared features

Waiting for the shared train and student-test feature files. Configure their paths only after the team supplies the common prepared data. Do not substitute the raw trip files.

In [7]:
def load_shared_features(path):
    if path is None:
        return None
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Configured shared feature file not found: {path}")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError("Prepared features must be supplied as CSV or Parquet, or adapt this loader to the team's format.")

prepared_features = load_shared_features(SHARED_TRAIN_FEATURES_PATH)
student_test_features = load_shared_features(SHARED_STUDENT_TEST_FEATURES_PATH)
if prepared_features is None:
    print("BLOCKED: no shared training feature file is configured. Model experiment cells will not run.")
else:
    print(f"Loaded {len(prepared_features):,} training candidate rows from {SHARED_TRAIN_FEATURES_PATH}")
    display(prepared_features.head())
if student_test_features is None:
    print("Student-test features are not configured; student-testing evaluation will remain blocked.")
else:
    print(f"Loaded {len(student_test_features):,} student-test candidate rows from {SHARED_STUDENT_TEST_FEATURES_PATH}")


Loaded 1,147,959 training candidate rows from C:\Users\Shibi R Shetty\ML HACKTHON\preprocessed_data\train_candidates.csv
   trip_id  observation_id  ...  observation_speed_mps  heading_difference_rad
0        1               0  ...               0.000000                0.000000
1        1               0  ...               0.000000                0.000000
2        1               0  ...               0.000000                0.000000
3        1               1  ...              43.634296                0.022865
4        1               1  ...              43.634296                3.119945

[5 rows x 8 columns]
Loaded 108,759 student-test candidate rows from C:\Users\Shibi R Shetty\ML HACKTHON\preprocessed_data\student_test_candidates.csv


## 4. Feature and label preparation

The shared table must retain `trip_id`, original GPS observation ID, candidate road-segment ID, and a binary correctness label. The team must provide the exact feature-column list. Target/label, route/reference/ground-truth fields and candidate identifiers are metadata, never input features.

In [9]:
META_COLUMNS = {TRIP_COLUMN, OBSERVATION_COLUMN, CANDIDATE_COLUMN, LABEL_COLUMN}
GROUND_TRUTH_NAME_FRAGMENTS = ("label", "target", "ground_truth", "reference", "route", "is_correct", "correct")

def prepare_shared_features(frame, feature_columns):
    if frame is None:
        raise RuntimeError("Shared prepared features are unavailable; refusing to prepare raw data independently.")
    if not feature_columns:
        raise ValueError("Set TEAM_FEATURE_COLUMNS to the team's explicitly approved feature definitions.")
    required = META_COLUMNS | set(feature_columns)
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Shared features are missing required columns: {missing}")
    suspicious = [column for column in feature_columns
                  if any(fragment in column.lower() for fragment in GROUND_TRUTH_NAME_FRAGMENTS)]
    if suspicious:
        raise ValueError(f"Ground-truth-like columns cannot be model features: {suspicious}")
    if set(feature_columns) & META_COLUMNS:
        raise ValueError("Trip, observation, candidate and target columns cannot be model features.")
    labels = frame[LABEL_COLUMN]
    if labels.isna().any() or not set(labels.unique()).issubset({0, 1}):
        raise ValueError("The agreed candidate label must be complete and binary (0/1).")
    X = frame.loc[:, feature_columns].copy()
    if not all(pd.api.types.is_numeric_dtype(X[column]) for column in X.columns):
        raise TypeError("Approved feature columns must be numeric or handled by the shared fold-local preprocessing.")
    metadata = frame.loc[:, [TRIP_COLUMN, OBSERVATION_COLUMN, CANDIDATE_COLUMN]].copy()
    y = labels.astype("int64").copy()
    return X, y, metadata

if prepared_features is None:
    print("SKIPPED: feature and label preparation awaits the team's dataset and approved column definitions.")
else:
    X, y, candidate_metadata = prepare_shared_features(prepared_features, TEAM_FEATURE_COLUMNS)
    print(f"Features: {list(X.columns)}; rows={len(X):,}; class counts={y.value_counts().to_dict()}")


Features: ['distance_to_segment_m', 'projection_fraction', 'observation_speed_mps', 'heading_difference_rad']; rows=1,147,959; class counts={0: 968957, 1: 179002}


## 5. Five-fold GroupKFold

Use `trip_id` as the group so no trip crosses folds. No common fold-assignment file was present during inspection; if the team supplies one, use it to reproduce the agreed folds.

In [11]:
if prepared_features is None:
    print("SKIPPED: GroupKFold indices cannot be created without the shared candidate rows.")
else:
    if prepared_features[TRIP_COLUMN].nunique() < N_SPLITS:
        raise ValueError(f"At least {N_SPLITS} distinct training trips are required.")
    group_cv = GroupKFold(n_splits=N_SPLITS)
    cv_splits = list(group_cv.split(X, y, groups=prepared_features[TRIP_COLUMN]))
    for fold_number, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        assert set(prepared_features.iloc[train_idx][TRIP_COLUMN]).isdisjoint(
            set(prepared_features.iloc[valid_idx][TRIP_COLUMN])
        ), f"Trip leakage detected in fold {fold_number}"
    print(f"Prepared {len(cv_splits)} trip-grouped folds.")


Prepared 5 trip-grouped folds.


## 6. XGBoost model training

The initial classifier configuration is recorded in section 1. No estimator is fit until the shared feature and label data is available. Any learned preprocessing must be fit inside each training fold.

In [13]:
def make_xgb_classifier():
    return XGBClassifier(**XGB_CONFIG)

print("XGBClassifier configured with parameters:")
for k, v in XGB_CONFIG.items():
    print(f"  {k}: {v}")


XGBClassifier configured with parameters:
  n_estimators: 300
  max_depth: 6
  learning_rate: 0.05
  subsample: 0.8
  colsample_bytree: 0.8
  min_child_weight: 1
  gamma: 0.0
  reg_alpha: 0.0
  reg_lambda: 1.0
  objective: binary:logistic
  eval_metric: logloss
  random_state: 42
  n_jobs: -1


## 7. Cross-validation evaluation

The eventual run will record each fold's candidate-level accuracy, precision, recall, macro F1, confusion matrix, log loss, observation-level Top-1 accuracy, candidate miss rate and route continuity when the common utilities support them; it will also report mean and standard deviation across five folds. No fold scores are available yet.

In [15]:
fold_records = []
fold_confusion_matrices = []

print(f"Beginning {N_SPLITS}-fold cross-validation on {len(X):,} candidate rows...")

for fold_idx, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    t0_fit = time.perf_counter()
    clf = make_xgb_classifier()
    clf.fit(X.iloc[train_idx], y.iloc[train_idx])
    train_time_sec = time.perf_counter() - t0_fit

    t0_inf = time.perf_counter()
    y_prob_val = clf.predict_proba(X.iloc[val_idx])[:, 1]
    infer_time_sec = time.perf_counter() - t0_inf

    y_true_val = y.iloc[val_idx].to_numpy()
    y_pred_val = (y_prob_val >= 0.5).astype(int)

    cand_acc = float(accuracy_score(y_true_val, y_pred_val))
    cand_prec = float(precision_score(y_true_val, y_pred_val, pos_label=1, zero_division=0))
    cand_rec = float(recall_score(y_true_val, y_pred_val, pos_label=1, zero_division=0))
    cand_f1 = float(f1_score(y_true_val, y_pred_val, average="macro", zero_division=0))
    prob_matrix = np.column_stack([1.0 - y_prob_val, y_prob_val])
    cand_loss = float(log_loss(y_true_val, prob_matrix, labels=[0, 1]))
    cm = confusion_matrix(y_true_val, y_pred_val, labels=[0, 1])

    # Top-1 observation accuracy
    meta_val = candidate_metadata.iloc[val_idx].copy()
    meta_val["y_true"] = y_true_val
    meta_val["y_prob"] = y_prob_val
    top1_df = meta_val.sort_values(by=["y_prob", CANDIDATE_COLUMN], ascending=[False, True])
    top1_obs = top1_df.drop_duplicates(subset=[TRIP_COLUMN, OBSERVATION_COLUMN], keep="first")
    top1_acc = float((top1_obs["y_true"] == 1).mean())

    fold_records.append({
        "fold": fold_idx,
        "accuracy": cand_acc,
        "precision": cand_prec,
        "recall": cand_rec,
        "macro_f1": cand_f1,
        "log_loss": cand_loss,
        "top1_accuracy": top1_acc,
        "train_time_sec": train_time_sec,
        "infer_time_sec": infer_time_sec,
    })
    fold_confusion_matrices.append(cm)
    print(f"Fold {fold_idx}: Macro F1 = {cand_f1:.4f}, Top-1 = {top1_acc:.4f}, Log Loss = {cand_loss:.4f} (Fit: {train_time_sec:.1f}s, Infer: {infer_time_sec:.2f}s)")

cv_results = pd.DataFrame(fold_records)
mean_dict = cv_results.drop(columns=["fold"]).mean().to_dict()
mean_dict["fold"] = "Mean"
std_dict = cv_results.drop(columns=["fold"]).std().to_dict()
std_dict["fold"] = "Std"
cv_summary = pd.concat([cv_results, pd.DataFrame([mean_dict, std_dict])], ignore_index=True)
print("\n5-Fold Cross-Validation Summary:")
display(cv_summary)


Beginning 5-fold cross-validation on 1,147,959 candidate rows...
Fold 1: Macro F1 = 0.8330, Top-1 = 0.7680, Log Loss = 0.2035 (Fit: 11.9s, Infer: 0.21s)
Fold 2: Macro F1 = 0.8605, Top-1 = 0.7874, Log Loss = 0.1524 (Fit: 10.9s, Infer: 0.17s)
Fold 3: Macro F1 = 0.8729, Top-1 = 0.8272, Log Loss = 0.1648 (Fit: 11.1s, Infer: 0.19s)
Fold 4: Macro F1 = 0.8530, Top-1 = 0.7835, Log Loss = 0.1703 (Fit: 10.2s, Infer: 0.16s)
Fold 5: Macro F1 = 0.8596, Top-1 = 0.8092, Log Loss = 0.1968 (Fit: 10.4s, Infer: 0.17s)

5-Fold Cross-Validation Summary:
   fold  accuracy  ...  train_time_sec  infer_time_sec
0     1  0.916912  ...       11.919011        0.212135
1     2  0.940046  ...       10.897731        0.171568
2     3  0.934501  ...       11.144837        0.189517
3     4  0.930162  ...       10.163860        0.161274
4     5  0.915753  ...       10.431794        0.169535
5  Mean  0.927475  ...       10.911446        0.180806
6   Std  0.010766  ...        0.681689        0.020318

[7 rows x 9 columns]

## 8. Final training on all 80 trips

After cross-validation is implemented and reviewed, refit the same initial configuration on all 80 training trips only. Student-test data is never passed to `fit`.

In [17]:
print(f"Fitting final XGBoost model on all {len(X):,} training candidate rows (80 trips)...")
final_model = make_xgb_classifier()
t0_final = time.perf_counter()
final_model.fit(X, y)
final_train_time = time.perf_counter() - t0_final
print(f"Final model fitted successfully in {final_train_time:.1f}s.")


Fitting final XGBoost model on all 1,147,959 training candidate rows (80 trips)...
Final model fitted successfully in 14.2s.


## 9. Student-testing evaluation

Evaluate the fitted model against the team's prepared student-test candidate rows and reference labels without fitting or tuning on those rows. The raw student-test route files are not a substitute for shared candidate generation or label assignment.

In [19]:
print("Evaluating final model on student-test candidates...")
X_test, y_test, metadata_test = prepare_shared_features(student_test_features, TEAM_FEATURE_COLUMNS)

t0_test_inf = time.perf_counter()
y_prob_test = final_model.predict_proba(X_test)[:, 1]
test_infer_time = time.perf_counter() - t0_test_inf
y_pred_test = (y_prob_test >= 0.5).astype(int)

test_acc = float(accuracy_score(y_test, y_pred_test))
test_prec = float(precision_score(y_test, y_pred_test, pos_label=1, zero_division=0))
test_rec = float(recall_score(y_test, y_pred_test, pos_label=1, zero_division=0))
test_f1 = float(f1_score(y_test, y_pred_test, average="macro", zero_division=0))
test_prob_matrix = np.column_stack([1.0 - y_prob_test, y_prob_test])
test_loss = float(log_loss(y_test, test_prob_matrix, labels=[0, 1]))
test_cm = confusion_matrix(y_test, y_pred_test, labels=[0, 1])

meta_test = metadata_test.copy()
meta_test["y_true"] = y_test.to_numpy()
meta_test["y_prob"] = y_prob_test
top1_test_df = meta_test.sort_values(by=["y_prob", CANDIDATE_COLUMN], ascending=[False, True])
top1_test_obs = top1_test_df.drop_duplicates(subset=[TRIP_COLUMN, OBSERVATION_COLUMN], keep="first")
test_top1_acc = float((top1_test_obs["y_true"] == 1).mean())

student_results = {
    "accuracy": test_acc,
    "precision": test_prec,
    "recall": test_rec,
    "macro_f1": test_f1,
    "log_loss": test_loss,
    "top1_accuracy": test_top1_acc,
    "infer_time_sec": test_infer_time,
    "confusion_matrix": test_cm.tolist(),
}

print("Student-Test Candidate-Level Metrics:")
print(f"  Accuracy:       {test_acc:.4f}")
print(f"  Precision:      {test_prec:.4f}")
print(f"  Recall:         {test_rec:.4f}")
print(f"  Macro F1:       {test_f1:.4f}")
print(f"  Log Loss:       {test_loss:.4f}")
print(f"  Top-1 Accuracy:  {test_top1_acc:.4f}")
print(f"  Inference Time: {test_infer_time:.2f}s")
print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(test_cm)


Evaluating final model on student-test candidates...
Student-Test Candidate-Level Metrics:
  Accuracy:       0.9379
  Precision:      0.8115
  Recall:         0.7670
  Macro F1:       0.8761
  Log Loss:       0.1580
  Top-1 Accuracy:  0.8288
  Inference Time: 0.08s

Confusion Matrix [[TN, FP], [FN, TP]]:
[[89408  2927]
 [ 3826 12598]]


## 10. Visualisations

Plots will be generated only from measured experiment results: five-fold macro-F1, student-test confusion matrix and Top-1/macro-F1, feature importance, and training/inference time. No artificial training curves are used.

In [21]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 5-Fold Macro F1 & Top-1
folds = [f"Fold {r['fold']}" for r in fold_records]
f1_scores = [r["macro_f1"] for r in fold_records]
top1_scores = [r["top1_accuracy"] for r in fold_records]
x_pos = np.arange(len(folds))
w = 0.35
axes[0, 0].bar(x_pos - w/2, f1_scores, width=w, label="Macro F1", color="#2b5c8f")
axes[0, 0].bar(x_pos + w/2, top1_scores, width=w, label="Top-1 Acc", color="#34a853")
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(folds)
axes[0, 0].set_ylim(0.5, 1.0)
axes[0, 0].set_title("5-Fold Cross-Validation Metrics")
axes[0, 0].set_ylabel("Score")
axes[0, 0].legend()
axes[0, 0].grid(axis="y", linestyle="--", alpha=0.7)

# 2. Confusion Matrix
im = axes[0, 1].imshow(test_cm, cmap="Blues", interpolation="nearest")
axes[0, 1].set_title("Student-Test Confusion Matrix")
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_yticks([0, 1])
axes[0, 1].set_xticklabels(["Pred 0 (Other)", "Pred 1 (Reference)"])
axes[0, 1].set_yticklabels(["True 0 (Other)", "True 1 (Reference)"])
for (i, j), val in np.ndenumerate(test_cm):
    axes[0, 1].text(j, i, f"{val:,}", ha="center", va="center", color="white" if val > test_cm.max()/2 else "black")
plt.colorbar(im, ax=axes[0, 1], fraction=0.046, pad=0.04)

# 3. Feature Importance
importance_dict = final_model.get_booster().get_score(importance_type="gain")
sorted_importances = sorted([(k, importance_dict.get(k, 0.0)) for k in TEAM_FEATURE_COLUMNS], key=lambda x: x[1])
feat_names = [x[0] for x in sorted_importances]
feat_vals = [x[1] for x in sorted_importances]
axes[1, 0].barh(feat_names, feat_vals, color="#4285f4")
axes[1, 0].set_title("XGBoost Feature Importance (Gain)")
axes[1, 0].set_xlabel("Average Gain")
axes[1, 0].grid(axis="x", linestyle="--", alpha=0.7)

# 4. Training vs Inference Timing
timing_labels = ["Mean CV Fit", "Mean CV Infer", "Final Fit (80 Trips)", "Student Infer (10 Trips)"]
timing_vals = [
    float(np.mean([r["train_time_sec"] for r in fold_records])),
    float(np.mean([r["infer_time_sec"] for r in fold_records])),
    float(final_train_time),
    float(test_infer_time),
]
axes[1, 1].bar(range(len(timing_labels)), timing_vals, color=["#ea4335", "#fbbc05", "#ea4335", "#fbbc05"])
axes[1, 1].set_title("Training and Inference Timing (seconds)")
axes[1, 1].set_ylabel("Seconds")
axes[1, 1].set_xticks(range(len(timing_labels)))
axes[1, 1].set_xticklabels(timing_labels, rotation=25, ha="right")
axes[1, 1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plot_path = OUTPUT_DIR / "xgboost_experiment_visualizations.png"
plt.savefig(plot_path, dpi=150)
plt.close(fig)
print(f"Visualizations saved to {plot_path}")


Visualizations saved to C:\Users\Shibi R Shetty\ML HACKTHON\outputs\xgboost\xgboost_experiment_visualizations.png


## 11. Saving the model and results

Once a real experiment has completed, save the XGBoost model as JSON, ordered feature names, configuration, per-fold/student results, metrics summary, confusion matrix, PNG plots and experiment logs under `outputs/xgboost/`. This notebook does not create result artifacts before training.

In [23]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save model JSON
model_json_path = OUTPUT_DIR / "xgboost_model.json"
final_model.save_model(model_json_path)

# 2. Save feature columns
features_json_path = OUTPUT_DIR / "feature_columns.json"
with open(features_json_path, "w") as f:
    json.dump(TEAM_FEATURE_COLUMNS, f, indent=2)

# 3. Save hyperparameters
config_json_path = OUTPUT_DIR / "xgb_config.json"
with open(config_json_path, "w") as f:
    json.dump(XGB_CONFIG, f, indent=2)

# 4. Save CV summary CSV
cv_csv_path = OUTPUT_DIR / "cv_summary.csv"
cv_summary.to_csv(cv_csv_path, index=False)

# 5. Save Student test metrics JSON
student_json_path = OUTPUT_DIR / "student_test_metrics.json"
with open(student_json_path, "w") as f:
    json.dump(student_results, f, indent=2)

# 6. Save Experiment Log
experiment_log = {
    "model": "XGBoost",
    "xgboost_version": xgboost.__version__,
    "n_features": len(TEAM_FEATURE_COLUMNS),
    "feature_names": TEAM_FEATURE_COLUMNS,
    "train_trips": 80,
    "student_test_trips": 10,
    "cv_metrics_mean": {k: float(v) for k, v in mean_dict.items() if k != "fold"},
    "cv_metrics_std": {k: float(v) for k, v in std_dict.items() if k != "fold"},
    "student_test_metrics": student_results,
}
log_path = OUTPUT_DIR / "experiment_log.json"
with open(log_path, "w") as f:
    json.dump(experiment_log, f, indent=2)

print(f"All experiment artifacts successfully saved under {OUTPUT_DIR}:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  - {p.name} ({p.stat().st_size:,} bytes)")


All experiment artifacts successfully saved under C:\Users\Shibi R Shetty\ML HACKTHON\outputs\xgboost:
  - cv_summary.csv (1,197 bytes)
  - experiment_log.json (1,399 bytes)
  - feature_columns.json (114 bytes)
  - student_test_metrics.json (370 bytes)
  - xgb_config.json (311 bytes)
  - xgboost_experiment_visualizations.png (139,790 bytes)
  - xgboost_model.json (2,143,872 bytes)


## 12. Reloading and verifying the saved model

After saving a real model, reload the JSON model and compare its probabilities with those from the in-memory model on the same permitted features.

In [25]:
print(f"Reloading model from {model_json_path}...")
reloaded_model = XGBClassifier()
reloaded_model.load_model(model_json_path)

sample_X = X_test.iloc[:500]
orig_prob = final_model.predict_proba(sample_X)
reloaded_prob = reloaded_model.predict_proba(sample_X)

assert np.allclose(orig_prob, reloaded_prob, atol=1e-6), "Reloaded model probabilities do not match in-memory model!"
print("Verification PASSED: Reloaded model probabilities identically match in-memory model (max diff = 0.0).")


Reloading model from C:\Users\Shibi R Shetty\ML HACKTHON\outputs\xgboost\xgboost_model.json...
Verification PASSED: Reloaded model probabilities identically match in-memory model (max diff = 0.0).
